# Required Libraries and Functions

In [0]:
from pyspark.sql.functions import col, current_timestamp, broadcast

In [0]:
trips_df = spark.read.table("workspace.taxi_weather.silver_trips")
zone_df = spark.read.table("workspace.taxi_weather.silver_zone")
weather_df = spark.read.table("workspace.taxi_weather.silver_weather")

clean_trips = (trips_df.alias("t")
                 .join(broadcast(zone_df).alias("z"), how="left", on=col("t.PULocationID")==col("z.LocationID"))
                 .join(broadcast(weather_df).alias("w"), how="left", on=((col("t.pickup_datehour")==col("w.weather_datetime")) & (col("z.borough_id")==col("w.borough_id"))))
                 )

gold_fact_trips = (
    clean_trips.select(
        col("pickup_datehour"),
        col("dropoff_datehour"),
        col("tpep_pickup_datetime"),
        col("tpep_dropoff_datetime"),
        col("PULocationID"),
        col("DOLocationID"),
        col("passenger_count"),
        col("trip_distance"),
        col("store_and_fwd_flag"),
        col("VendorID"),
        col("RatecodeID"),
        col("payment_type"),
        col("fare_amount"),
        col("extra"),
        col("mta_tax"),
        col("tip_amount"),
        col("tolls_amount"),
        col("improvement_surcharge"),
        col("total_amount"),
        col("congestion_surcharge"),
        col("Airport_fee"),
        col("cbd_congestion_fee"),

        col("z.borough_id").alias("pickup_borough_id"),

        col("w.weather_code").alias("pickup_weather_code"),
        col("w.temperature_2m").alias("pickup_temperature_2m"),
        col("w.precipitation").alias("pickup_precipitation"),
        col("w.snowfall").alias("pickup_snowfall")
        )
    .withColumn("gold_processed_time", current_timestamp())
    )

In [0]:
borough_df = spark.read.table("workspace.taxi_weather.silver_borough_dim")

vendor_df = spark.read.table("workspace.taxi_weather.silver_vendor_dim")
rate_df = spark.read.table("workspace.taxi_weather.silver_rate_dim")
payment_df = spark.read.table("workspace.taxi_weather.silver_payment_dim")
wc_df = spark.read.table("workspace.taxi_weather.silver_wc_dim")



clean_trips_view = (gold_fact_trips.alias("g")
                    .join(broadcast(zone_df).alias("zP")   , how="left", on=col("g.PULocationID")==col("zP.LocationID"))
                    .join(broadcast(borough_df).alias("bP"), how="left", on=col("g.pickup_borough_id")==col("bP.borough_id"))
                    .join(broadcast(zone_df).alias("zD")   , how="left", on=col("g.DOLocationID")==col("zD.LocationID"))
                    .join(broadcast(borough_df).alias("bD"), how="left", on=col("zD.borough_id")==col("bD.borough_id"))
                    .join(broadcast(vendor_df).alias("v")  , how="left", on=col("g.VendorID")==col("v.VendorID"))
                    .join(broadcast(rate_df).alias("r")    , how="left", on=col("g.RatecodeID")==col("r.RatecodeID"))
                    .join(broadcast(payment_df).alias("p") , how="left", on=col("g.payment_type")==col("p.payment_type"))
                    .join(broadcast(wc_df).alias("wc")     , how="left", on=col("g.pickup_weather_code")==col("wc.weather_code"))
                    )

# Create Denormalized One Big Table (OBT)

In [0]:
gold_view = (
    clean_trips_view.select(
        col("tpep_pickup_datetime"),
        col("tpep_dropoff_datetime"),

        col("bP.name").alias("pickup_borough"),
        col("zP.Zone").alias("pickup_zone"),
        col("zP.service_zone").alias("pickup_service_zone"),
        col("bD.name").alias("dropoff_borough"),
        col("zD.Zone").alias("dropoff_zone"),
        col("zD.service_zone").alias("dropoff_service_zone"),    

        col("passenger_count"),
        col("trip_distance"),
        col("store_and_fwd_flag"),

        col("v.name").alias("vendor_name"),

        col("r.description").alias("rate_description"),

        col("p.type").alias("payment_type"),

        col("fare_amount"),
        col("extra"),
        col("mta_tax"),
        col("tip_amount"),
        col("tolls_amount"),
        col("improvement_surcharge"),
        col("total_amount"),
        col("congestion_surcharge"),
        col("Airport_fee"),
        col("cbd_congestion_fee"),


        col("wc.description").alias("pickup_weather_description"),

        col("pickup_temperature_2m"),
        col("pickup_precipitation"),
        col("pickup_snowfall"),

        col("dropoff_datehour")
        )
    )

In [0]:
gold_fact_trips.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.taxi_weather.gold_facts")

gold_view.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.taxi_weather.gold_obt_trips")